# TARDIS Config Report
Auto-generated prototype notebook for a single config.

In [ ]:
# Parameters
config_path = globals().get("config_path", None)  # Path to TARDIS config file
atom_data = globals().get("atom_data", "kurucz_cd23_chianti_H_He_latest")  # Atom data identifier

In [ ]:
from pathlib import Path
from tardis import run_tardis
from tardis.io.configuration.config_reader import Configuration
from tardis.visualization import (
    SDECPlotter,
    LIVPlotter,
    LineInfoWidget,
    GrotrianWidget,
    CustomAbundanceWidget,
    shell_info_from_simulation,
)

In [ ]:
config = Configuration.from_yaml(config_path)
try:
    config.montecarlo.tracking.track_rpacket = True
except Exception:
    pass
try:
    sim = run_tardis(config, atom_data=atom_data, virtual_packet_logging=True, show_convergence_plots=True)
except RuntimeError as exc:
    if 'Convergence Plots cannot be displayed in command-line' not in str(exc):
        raise
    sim = run_tardis(config, atom_data=atom_data, virtual_packet_logging=True, show_convergence_plots=False)
print('Simulation complete for', Path(config_path).name)

## Interactive Widgets (Optional)
These widget cells use TARDIS visualization widgets when available and skip gracefully in non-interactive environments.

In [ ]:
widgets = {}

try:
    widgets['shell_info'] = shell_info_from_simulation(sim)
    print('Shell Info widget ready')
except Exception as exc:
    print(f'Shell Info widget unavailable: {exc}')

try:
    widgets['line_info'] = LineInfoWidget.from_simulation(sim)
    print('Line Info widget ready')
except Exception as exc:
    print(f'Line Info widget unavailable: {exc}')

try:
    widgets['grotrian'] = GrotrianWidget.from_simulation(sim)
    print('Grotrian widget ready')
except Exception as exc:
    print(f'Grotrian widget unavailable: {exc}')

try:
    widgets['custom_abundance'] = CustomAbundanceWidget.from_yml(config_path)
    print('Custom Abundance widget ready')
except Exception as exc:
    print(f'Custom Abundance widget unavailable: {exc}')

In [ ]:
for widget_name, widget_obj in widgets.items():
    print(f'Displaying widget: {widget_name}')
    try:
        widget_obj.display()
    except Exception as exc:
        print(f'Could not render {widget_name} in this environment: {exc}')

## SDEC Visualizations

In [ ]:
sdec = SDECPlotter.from_simulation(sim)
sdec.generate_plot_mpl(packets_mode='real')

In [ ]:
sdec.generate_plot_mpl(packets_mode='virtual')

## LIV Visualizations

In [ ]:
liv = LIVPlotter.from_simulation(sim)
liv.generate_plot_mpl(packets_mode='real')

In [ ]:
liv.generate_plot_mpl(packets_mode='virtual')

## Convergence Visualizations (If Available)
Convergence plots are shown when available in the simulation object.

In [ ]:
try:
    if hasattr(sim, 'convergence_plots') and sim.convergence_plots is not None:
        sim.convergence_plots.plasma_plot.show(renderer='notebook_connected')
        sim.convergence_plots.t_inner_luminosities_plot.show(renderer='notebook_connected')
    else:
        print('Convergence plots are not present. Enable show_convergence_plots=True in run_tardis to generate them.')
except Exception as exc:
    print(f'Could not display convergence plots: {exc}')